# NHS A&E Performance Analysis
**Author:** Husnain Zahoor  
**Date:** Aug 05, 2026  
**Dataset:** NHS AE Quality Indicators (Provisional, December 2023 – December 2025)  
**Source URL:** https://tinyurl.com/NHS-Source-Data

---

### Context & Pipeline Position
This notebook (4 of 4) automates the generation of a boardroom-ready, 7-sheet Excel workbook (`NHS_AE_Report.xlsx`) using `openpyxl`. It formats title sheets, data flow documentation, wrangling logs, derived metrics, visual layouts, and executive recommendations.

## Environment Setup
Mount Google Drive and define standard file paths. Run these three cells at the start of every session.

In [ ]:
# Mount Google Drive — run this first in every session
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define standard paths — reference these throughout the module
DATA_PATH   = '/content/drive/My Drive/Lumen/python-data-analytics/Data/'
OUTPUT_PATH = '/content/drive/My Drive/Lumen/python-data-analytics/Output/'

In [ ]:
# Verify setup — confirm all three CSV files are accessible
import os
print(os.listdir(DATA_PATH))

['aeqi_metadata.csv', 'aeqi_open_data_2025_12.csv', 'nhs_trust_reference.csv']


## Library Imports

In [ ]:
# Setup and Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
import os

# FORCE PANDAS TO SHOW ALL COLUMNS HORIZONTALLY (NO WRAPPING)
pd.set_option('display.max_columns', None)  # Show every column
pd.set_option('display.width', 1000)        # Give the text engine plenty of horizontal width

##Build the Excel report skeleton.

In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import pandas as pd

# Step 1: Key Constants used accros analysis
NHS_SOURCE = 'Source: NHS England, Provisional A&E Quality Indicators, December 2025'
REPORT_TITLE = 'NHS A&E Quality Indicators Analysis, Dec 2023 to Dec 2025'
ANALYST_NAME = 'Husnain Zahoor'
OUTPUT_PATH = '/content/drive/My Drive/Lumen/python-data-analytics/Output/'

# Step 2: create workbook + 7 sheets
wb = Workbook()
# Rename the default sheet and add the rest
sheet_names = [
    '1. Title Page', '2. Population Flow', '3. Consistency Checks',
    '4. Wrangling Steps', '5. Column Derivations', '6. Visualisations',
    '7. Recommendations'
]
# Rename the auto-created first sheet
wb.active.title = sheet_names[0]

# Create the remaining six sheets
for name in sheet_names[1:]:
    wb.create_sheet(title=name)

print([ws.title for ws in wb.worksheets])

['1. Title Page', '2. Population Flow', '3. Consistency Checks', '4. Wrangling Steps', '5. Column Derivations', '6. Visualisations', '7. Recommendations']


##Populate the Sheet 1: **Title Page**

In [ ]:
# Step 3: write the title page
ws_title = wb['1. Title Page']
ws_title['B2'] = REPORT_TITLE
ws_title['B2'].font = Font(bold=True, size=18, color='1F4E79')
ws_title['B4'] = 'Analyst:'
ws_title['C4'] = ANALYST_NAME
ws_title['B5'] = 'Date:'
ws_title['C5'] = '27 July 2026'
ws_title['B6'] = 'Data source:'
ws_title['C6'] = NHS_SOURCE
ws_title['B8'] = 'This report contains:'
ws_title['B9'] = '— Data quality and consistency checks (Sheet 3)'
ws_title['B10'] = '— Wrangling decisions and population flow (Sheets 2, 4)'
ws_title['B11'] = '— Derived variable specifications (Sheet 5)'
ws_title['B12'] = '— Visualisations (Sheet 6)'
ws_title['B13'] = '— Findings and recommendations (Sheet 7)'
# Set column width
ws_title.column_dimensions['B'].width = 22
ws_title.column_dimensions['C'].width = 65

In [ ]:
# Step 4: save
REPORT_PATH = OUTPUT_PATH + 'NHS_AE_Report.xlsx'
wb.save(REPORT_PATH)
print('Saved:', REPORT_PATH)

Saved: /content/drive/My Drive/Lumen/python-data-analytics/Output/NHS_AE_Report.xlsx


##Populate Sheets 2–5.

In [ ]:
population_flow = pd.DataFrame({
    'Stage': [
        'Source file (all orgs, all months)',
        'Suppressed rows identified (SUPPRESSION = Y)',
        'After removing ENGLAND aggregate (Lesson 4)',
        'After Join 1: + KPI metadata (left join on MEASURE_ID)',
        'After Join 2: + Trust reference (left join on ORG_CODE)',
        'After filtering to active NHS Trusts only',
    ],
    'Row Count': [
        94720,
        94720,
        94145,
        94145,
        94145,
        78386,
    ],
    'Notes': [
        '198 orgs, 23 KPIs, 25 months',
        '1,041 rows flagged (1.10%) retained in dataset, excluded from aggregations via SUPPRESSION != "Y" filter at analysis time, not dropped',
        '575 rows removed (25 months x 23 KPIs)',
        'Row count unchanged; 5 metadata columns added',
        '56 non-Trust orgs have NaN in REGION; retained at this stage',
        '140 active NHS Trusts with submissions; 65 active Trusts made no submissions in period; 56 non-Trust orgs and 68 closed Trusts excluded',
    ]
})

print(population_flow.to_string(index=False))

                                                  Stage  Row Count                                                                                                                                   Notes
                     Source file (all orgs, all months)      94720                                                                                                            198 orgs, 23 KPIs, 25 months
           Suppressed rows identified (SUPPRESSION = Y)      94720  1,041 rows flagged (1.10%) retained in dataset, excluded from aggregations via SUPPRESSION != "Y" filter at analysis time, not dropped
            After removing ENGLAND aggregate (Lesson 4)      94145                                                                                                  575 rows removed (25 months x 23 KPIs)
 After Join 1: + KPI metadata (left join on MEASURE_ID)      94145                                                                                           Row count unchanged; 5 metadata

In [ ]:
consistency_log = pd.DataFrame({
    'Check': [
        'Mixed types SUPPRESSION column',
        'Mixed types MEASURE_VALUE',
        'Negative values in MEASURE_VALUE',
        'Missing values (NaN) ORG_NAME',
        'Left join unmatched rows Join 2',
        'Suppressed rows (SUPPRESSION = Y)',
        'Exact duplicate rows',
        'Composite key duplicates',
        'Coverage vs theoretical maximum',
        'Analytical population df_final',
    ],
    'Finding': [
        'Mixed str/float due to blank CSV cells read as NaN',
        'Consistently float64; mixes counts, percentages, and wait times in same column',
        '62 rows with negative values in AEQI041 and AEQI031 at Bradford Teaching Hospitals, Sherwood Forest, Ashford & St Peters, Surrey and Sussex',
        '1,080 genuine NaN values in ORG_NAME in df_aeqi',
        '15,486 rows (56 unique org codes) returned NaN in REGION after Join 2 providers not registered as NHS Trusts (UTCs, GP cooperatives, ICBs)',
        '1,041 rows (1.10% of dataset); concentrated in ambulance denominators and low-volume providers',
        '0 exact duplicates confirmed across 94,145 rows',
        '0 composite key duplicates; ORG_CODE + ATTENDANCE_MONTH + MEASURE_ID is unique',
        '94,145 of 113,275 theoretical rows present (83.1% coverage); 16.9% shortfall',
        '78,386 rows; 140 active NHS Trusts with submissions; 65 active Trusts in register made no submissions in period',
    ],
    'Action Taken': [
        'fillna("") applied; SUPPRESSION column now consistently str throughout',
        'No change; filter by MEASURE_ID before any aggregation',
        'Retained; flagged for stakeholder awareness; not imputed or removed',
        'Resolved in Lesson 6 via left join on ORG_CODE from df_trusts; ORG_NAME now populated for all matched Trust rows',
        'Investigated as business logic finding not a data quality problem; excluded from df_final using REGION.notna() filter',
        'Retained; excluded from numeric aggregations using SUPPRESSION != "Y" filter at point of analysis',
        'No action required',
        'No action required',
        'Documented as inherent to source data; missing submissions and design-excluded KPIs are separate causes',
        'Three filters applied: ENG aggregate removed, closed Trusts removed, non-Trust providers removed',
    ]
})

consistency_log.to_csv(OUTPUT_PATH + 'consistency_checks.csv', index=False)
print("Saved: consistency_checks.csv")
print()
print(consistency_log.to_string(index=False))

Saved: consistency_checks.csv

                            Check                                                                                                                                     Finding                                                                                                          Action Taken
   Mixed types SUPPRESSION column                                                                                          Mixed str/float due to blank CSV cells read as NaN                                                fillna("") applied; SUPPRESSION column now consistently str throughout
        Mixed types MEASURE_VALUE                                                              Consistently float64; mixes counts, percentages, and wait times in same column                                                                No change; filter by MEASURE_ID before any aggregation
 Negative values in MEASURE_VALUE 62 rows with negative values in AEQI041 and AEQI031 at Brad

In [ ]:
derivations_log = pd.DataFrame({
    'Column Name': [
        'is_time_measure',
        'breach_flag',
        'performance_band',
        'month_name',
        'is_winter',
        'reattendance_band',
        'regional_mean_wait',
        'vs_region',
        'regional_band',
        'national_rank',
        'regional_rank',
    ],
    'Source Columns': [
        'MEASURE_ID',
        'is_time_measure, MEASURE_VALUE',
        'is_time_measure, MEASURE_VALUE',
        'ATTENDANCE_MONTH',
        'ATTENDANCE_MONTH',
        'MEASURE_ID, MEASURE_VALUE',
        'REGION, MEASURE_VALUE',
        'MEASURE_VALUE, regional_mean_wait',
        'MEASURE_VALUE, regional_mean_wait',
        'AEQI051',
        'REGION, AEQI051',
    ],
    'Logic / Conditions': [
        'np.where + isin() True for KPIs where MEASURE_VALUE represents minutes; False for counts and rates',
        'np.where, if time measure AND MEASURE_VALUE > 240 then 1 else 0',
        '.loc[] with three conditions Good (<=200 min), Acceptable (200–240 min), Poor (>240 min); time measures only',
        '.dt.strftime() full month and year label for chart axis display (e.g. December 2024)',
        '.dt.month.isin() True for Oct–Mar (NHS winter definition); False for Apr–Sep',
        'user-defined function + .apply() AEQI062 only: Low (<8%), Medium (8–12%), High (>12%)',
        'groupby(REGION).transform(mean) regional benchmark mean wait time per NHS region',
        'MEASURE_VALUE minus regional_mean_wait gap in minutes; positive = worse than region',
        '.loc[] with two threshold conditions (±10%) Better than region (<90% of regional mean), Average, Worse than region (>110% of regional mean)',
        'rank(ascending=True, method=min) on AEQI051 rank 1 = shortest wait time (best)',
        'groupby(REGION).rank(ascending=True, method=min) on AEQI051 rank 1 = shortest wait time within that region',
    ],
    'Threshold Justification': [
        'N/A classification, no threshold',
        '240 min = NHS four-hour A&E standard (national policy benchmark)',
        '200 min / 240 min bands derived from the same four-hour standard',
        'N/A date formatting, no threshold',
        'Oct–Mar = standard NHS winter pressure definition',
        '8% / 12% = working benchmark bands confirm source before board use',
        'N/A descriptive statistic (mean), no threshold',
        'N/A arithmetic difference, no threshold',
        '±10% working assumption, not yet validated with clinical lead',
        'N/A ranking, no threshold',
        'N/A ranking, no threshold',
    ],
    'Applies To': [
        'All rows',
        'All rows (flag = 0 for non-time measures)',
        'Time-measure rows; NaN for non-time measures',
        'All rows',
        'All rows',
        'AEQI062 rows only; NaN for all other measures',
        'AEQI051 rows only',
        'AEQI051 rows only',
        'AEQI051 rows only',
        'Trust-level (pivot_trust), AEQI051 non-null rows only',
        'Trust-level (pivot_trust), AEQI051 non-null rows only, ranked within REGION',
    ]
})

print(derivations_log.to_string(index=False))
derivations_log.to_csv(OUTPUT_PATH + 'derivations_log.csv', index=False)
print("\nSaved: derivations_log.csv")

       Column Name                    Source Columns                                                                                                                          Logic / Conditions                                            Threshold Justification                                                                  Applies To
   is_time_measure                        MEASURE_ID                                          np.where + isin() True for KPIs where MEASURE_VALUE represents minutes; False for counts and rates                                   N/A classification, no threshold                                                                    All rows
       breach_flag    is_time_measure, MEASURE_VALUE                                                                             np.where, if time measure AND MEASURE_VALUE > 240 then 1 else 0   240 min = NHS four-hour A&E standard (national policy benchmark)                                   All rows (flag = 0 for non-time me

In [ ]:
# Append DataFrames using ExcelWriter
with pd.ExcelWriter(REPORT_PATH, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:

    # Sheet 2: Population Flow (From Lesson 6)
    population_flow.to_excel(writer, sheet_name='2. Population Flow', index=False, startrow=1)

    # Sheet 3: Consistency Checks (started Lesson 5, completed Lesson 7)
    consistency_log.to_excel(writer, sheet_name='3. Consistency Checks', index=False, startrow=1)

   # Sheet 4: Wrangling Steps (Trust ranking pivot table from Lesson 8)
    pivot_trust_ranked = pd.read_csv(OUTPUT_PATH + 'pivot_trust_ranked.csv')
    pivot_trust_ranked.to_excel(writer, sheet_name='4. Wrangling Steps', index=False, startrow=1)

    # Sheet 5: Column Derivations (started Lesson 7, completed Lesson 8)
    derivations_log.to_excel(writer, sheet_name='5. Column Derivations', index=False, startrow=1)

##Embed charts in Sheet 6.

In [ ]:
# Re-open the workbook to add images
wb2 = openpyxl.load_workbook(REPORT_PATH)
ws_charts = wb2['6. Visualisations']

chart_files = [
    ('chart_top10_wait_time.png',          'B2'),
    ('chart_england_trend.png',           'B32'),
    ('chart_reattendance_distribution.png', 'M2'),
    ('chart_treatment_vs_reattendance.png', 'M32'),
]

for filename, cell_anchor in chart_files:
    img = XLImage(OUTPUT_PATH + filename)
    img.width = 480
    img.height = 300
    ws_charts.add_image(img, cell_anchor)

wb2.save(REPORT_PATH)
print('Charts embedded. Report saved.')

Charts embedded. Report saved.


In [ ]:
findings = [
    ('B22', 'Blackpool Teaching Hospitals has the longest average A&E wait at 352 minutes, 83 minutes above the next worst Trust.'),
    ('B52', 'National A&E wait times peaked at 186 minutes in December 2023 and have stabilised in a tighter 159–167 minute range since spring 2025.'),
    ('M22', 'About ~27 Trusts have reattendance rates above 10%, forming a distinct high-risk group rather than random variation.'),
    ('M52', 'Faster treatment times show no meaningful link to lower reattendance rates, two Trusts with the fastest treatment times actually have the highest reattendance rates in the dataset.'),
]

for cell, text in findings:
    ws_charts[cell] = text

wb2.save(REPORT_PATH)
print('Findings added. Report saved.')

Findings added. Report saved.


##Write Sheet 7 Recommendations.

In [ ]:
# 1. Build the Recommendations DataFrame using your exact findings
recommendations_df = pd.DataFrame({
    '#': [1, 2, 3],
    'Finding': [
        'Blackpool Teaching Hospitals is a persistent outlier in total A&E waiting time.',
        'National A&E wait times have stabilised after a period of decline.',
        'A small group of Trusts show persistently high reattendance rates unrelated to treatment speed.'
    ],
    'Evidence': [
        'Mean AEQI051 = 352 min across 25 months (Dec 2023–Dec 2025), 83 min above next worst Trust (Wirral Universiy Teaching Hospital, 269 min), >2x national median (169 min).',
        'AEQI051 (England aggregate) fell from 186 min (Dec 2023) to low of 157 min (Aug 2024); holding in 159–167 min range since spring 2025.',
        'AEQI062 range: 0%–15.2% across 137 Trusts (~27 Trusts >10%). Northumbria (15.2%) & Maidstone (14.7%) highest despite fast treatment (37.3 min & 42.2 min); no correlation between AEQI041 and AEQI062.'
    ],
    'Proposed Action': [
        'Commission a focused operational review of Blackpool Teaching Hospitals to identify root causes of sustained excess wait time.',
        'Investigate what changed operationally from spring 2025 onward that allowed national wait times to stabilise in the 159–167 minute range, after the winter 2024/25 spike back to 180 minutes.',
        'Investigate the ~27 Trusts with reattendance rates above 10% to identify shared clinical or discharge-process causes, since speed of treatment is not the driver.'
    ]
})


# 2. Write DataFrame to Sheet 7 of the Excel report
REPORT_PATH = OUTPUT_PATH + 'NHS_AE_Report.xlsx'

with pd.ExcelWriter(REPORT_PATH, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    recommendations_df.to_excel(writer, sheet_name='7. Recommendations', index=False, startrow=1)

print("Sheet 7 updated and workbook saved successfully.")

Sheet 7 updated and workbook saved successfully.


#**END**